In [2]:
import pandas as pd
import numpy as np
pd.set_option('future.no_silent_downcasting', True)

In [3]:
url_google = "https://docs.google.com/spreadsheets/d/1KXvk6y6WcX-s3b7VZAwCYotDKs3WWY0NvMwccOLpr5Q/export?format=csv&gid=1983317899"
url_facebook = "https://docs.google.com/spreadsheets/d/1KXvk6y6WcX-s3b7VZAwCYotDKs3WWY0NvMwccOLpr5Q/export?format=csv&gid=2025154129"
url_orders = "https://docs.google.com/spreadsheets/d/1u7TNCUlMPfIB2SsbjIbmJEnwpVuHo6FD_vQ-vvpCIIc/export?format=csv&gid=462836816"

In [4]:
# TRATAMENTO DOS DADOS GOOGLE ADS

df_google = pd.read_csv(url_google,skiprows=2)

df_google['Day'] = pd.to_datetime(df_google['Day'])

df_google_agrupado = df_google.groupby('Day',as_index=True).agg({
    'Cost':'sum',
    'Revenue':'sum',
    'Clicks':'sum',
    'Orders':'sum',
    'Impr.':'sum',
    'Conversions':'sum',
    'Views':'sum',
    'Engagements':'sum'
})

df_google_agrupado.reset_index(inplace=True)

df_google_agrupado['ROAS'] = df_google_agrupado['Revenue'] / df_google_agrupado['Cost']

df_google_agrupado = df_google_agrupado.rename(columns=lambda col: col + "_google")

In [5]:
# TRATAMENTO DOS DADOS FACEBOOK ADS

df_facebook = pd.read_csv(url_facebook)

df_facebook['Dia'] = pd.to_datetime(df_facebook['Dia'])

df_facebook_agrupado = df_facebook.groupby('Dia',as_index=True).agg({
    'Alcance':'sum',
    'Impressões':'sum',
    'Valor usado (BRL)':'sum',
    'Cliques no link':'sum',
    'Valor de conversão da compra':'sum'
})

df_facebook_agrupado.reset_index(inplace=True)

df_facebook_agrupado['ROAS'] = df_facebook_agrupado['Valor de conversão da compra'] / df_facebook_agrupado['Valor usado (BRL)']

df_facebook_agrupado = df_facebook_agrupado.rename(columns=lambda col: col + "_facebook")

In [6]:
print(df_google_agrupado.columns)
print(df_facebook_agrupado.columns)

Index(['Day_google', 'Cost_google', 'Revenue_google', 'Clicks_google',
       'Orders_google', 'Impr._google', 'Conversions_google', 'Views_google',
       'Engagements_google', 'ROAS_google'],
      dtype='object')
Index(['Dia_facebook', 'Alcance_facebook', 'Impressões_facebook',
       'Valor usado (BRL)_facebook', 'Cliques no link_facebook',
       'Valor de conversão da compra_facebook', 'ROAS_facebook'],
      dtype='object')


In [7]:
# TRATAMENTO DOS DADOS ORDERS (AGRUPAMENTO)

df_orders = pd.read_csv(url_orders)

#df_orders['Data'] = pd.to_datetime(df_orders['Data'],dayfirst=True)

colunas_para_converter = ['Faturamento líquido','Custo_BRL','Cotação_USD',
                          'Cotação_USD_Corrigida','Frete preço','Taxa de gateway','Taxa de checkout',
                          'Impostos','Custo de produto','Margem bruta']

df_orders[colunas_para_converter] = df_orders[colunas_para_converter].apply(lambda x: x.str.replace(',','.').astype(float))
#df_orders[colunas_para_converter] = df_orders[colunas_para_converter].apply(pd.to_numeric).astype(float)


df_orders_agrupado = df_orders.groupby('Data',as_index=True).agg({
    'Quantidade':'sum',
    'Faturamento líquido':'sum',
    'Custo_BRL':'sum',
    'Impostos':'sum',
    'Cotação_USD':'first',
    'Cotação_USD_Corrigida':'first',
    'Frete preço':'sum',
    'Taxa de gateway':'sum',
    'Taxa de checkout':'sum',
    'Custo de produto':'sum',
    'Margem bruta':'sum'
})

df_orders_agrupado.reset_index(inplace=True)

df_orders_agrupado.head()

,Data,Quantidade,Faturamento líquido,Custo_BRL,Impostos,Cotação_USD,Cotação_USD_Corrigida,Frete preço,Taxa de gateway,Taxa de checkout,Custo de produto,Margem bruta
0,01/01/2025,108,14564.98,7998.16,975.76,6.19,6.56,336.00,553.45,87.38,8973.99,5926.99
1,01/02/2025,145,21998.79,11153.00,1304.74,5.83,6.18,462.00,836.00,132.04,12457.76,10003.03
2,01/11/2024,279,34006.79,20580.21,1763.11,5.81,6.16,547.44,1292.26,204.01,22343.31,12210.92
3,01/12/2024,382,46547.95,27392.10,2611.59,6.05,6.42,751.88,1768.72,279.36,30003.74,17296.09
4,02/01/2025,236,23800.95,17739.44,1696.37,6.21,6.58,786.00,904.46,142.81,19435.81,5151.14


In [8]:
renomear_colunas_google = {
    'Day_google':'Data',
    'Cost_google':'Investimento Google Ads',
    'Revenue_google':'Receita Google Ads',
    'Clicks_google':'Clicks Google Ads',
    'Orders_google':'Pedidos Google Ads',
    'Impr._google':'Impressões Google Ads',
    'Conversions_google':'Conversões Google Ads',
    'Views_google':'Visualizações Google Ads',
    'Engagement_google':'Engajamentos Google Ads',
    'ROAS_google':'ROAS Google Ads'
}

renomear_colunas_facebook = {
    'Dia_facebook':'Data',
    'Alcance_facebook':'Alcance Facebook Ads',
    'Impressões_facebook':'Impressões Facebook Ads',
    'Valor usado (BRL)_facebook':'Investimento Facebook Ads',
    'Cliques no link_facebook':'Clicks Facebook Ads',
    'Visualizações_facebook':'Visualizações Facebook Ads',
    'Compras_facebook':'Compras Facebook Ads',
    'Valor de conversão da compra_facebook':'Receita Facebook Ads',
    'ROAS_facebook':'ROAS Facebook Ads'
}

In [9]:
df_google_agrupado = df_google_agrupado.rename(columns=renomear_colunas_google)
df_facebook_agrupado = df_facebook_agrupado.rename(columns=renomear_colunas_facebook)

In [10]:
df_ads = df_google_agrupado.merge(df_facebook_agrupado,on='Data',how='left')

In [11]:
df_ads.columns

Index(['Data', 'Investimento Google Ads', 'Receita Google Ads',
       'Clicks Google Ads', 'Pedidos Google Ads', 'Impressões Google Ads',
       'Conversões Google Ads', 'Visualizações Google Ads',
       'Engagements_google', 'ROAS Google Ads', 'Alcance Facebook Ads',
       'Impressões Facebook Ads', 'Investimento Facebook Ads',
       'Clicks Facebook Ads', 'Receita Facebook Ads', 'ROAS Facebook Ads'],
      dtype='object')

In [12]:
df_ads['Investimento em ads'] = df_ads['Investimento Google Ads'] + df_ads['Investimento Facebook Ads']
df_ads['Receita'] = df_ads['Receita Google Ads'] + df_ads['Receita Facebook Ads']
df_ads['ROAS'] = df_ads['Receita'] / df_ads['Investimento em ads'] 


df_ads['Data'] = df_ads['Data'].dt.strftime('%d/%m/%Y')

In [13]:
df_ads_final = df_orders_agrupado.merge(df_ads,on='Data',how='left')

df_ads_final['Lucro'] = df_ads_final['Faturamento líquido'] + df_ads_final['Frete preço'] - df_ads_final['Custo de produto'] - df_ads_final['Investimento em ads'] - df_ads_final['Taxa de gateway'] - df_ads_final['Taxa de checkout']


In [14]:
ProfitDB = df_ads_final.to_csv(r"C:\Users\GabrielCielo\Desktop\consolatio\Databases\ProfitDB.csv",float_format="%.2f",index=False,sep=';',decimal=',')

In [15]:
df_ads_final.head()

,Data,Quantidade,Faturamento líquido,Custo_BRL,Impostos,Cotação_USD,Cotação_USD_Corrigida,Frete preço,Taxa de gateway,Taxa de checkout,...,Alcance Facebook Ads,Impressões Facebook Ads,Investimento Facebook Ads,Clicks Facebook Ads,Receita Facebook Ads,ROAS Facebook Ads,Investimento em ads,Receita,ROAS,Lucro
0,01/01/2025,108,14564.98,7998.16,975.76,6.19,6.56,336.00,553.45,87.38,...,248215,284158,4238.43,4421.0,12785.64,3.016598,4787.85,19054.79,3.979822,498.31
1,01/02/2025,145,21998.79,11153.00,1304.74,5.83,6.18,462.00,836.00,132.04,...,200361,237820,4848.98,4843.0,20701.09,4.269164,5662.66,29277.19,5.170219,3372.33
2,01/11/2024,279,34006.79,20580.21,1763.11,5.81,6.16,547.44,1292.26,204.01,...,211889,246960,6289.87,5352.0,27539.25,4.378350,7504.59,32666.20,4.352829,3210.06
3,01/12/2024,382,46547.95,27392.10,2611.59,6.05,6.42,751.88,1768.72,279.36,...,129233,153943,3405.48,2802.0,32509.65,9.546275,4692.73,45258.47,9.644380,10555.28
4,02/01/2025,236,23800.95,17739.44,1696.37,6.21,6.58,786.00,904.46,142.81,...,219681,250870,3754.47,3839.0,15917.49,4.239610,4530.11,24376.48,5.380991,-426.24
